## Creating RAG 
## Initial Phase : Importing necessary modules 

In [1]:
%pip install langchain langchain-community langchain-chroma pypdf sentence-transformers

Defaulting to user installation because normal site-packages is not writeable
Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 25.3 -> 26.2.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [2]:
%pip install langchain-core langchain-ollama

Defaulting to user installation because normal site-packages is not writeable
Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 25.3 -> 26.2.1
[notice] To update, run: python.exe -m pip install --upgrade pip


## PHASE 1 : Document Loader

In [3]:
%pip install -U langchain-community pypdf

Defaulting to user installation because normal site-packages is not writeable
Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 25.3 -> 26.2.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [4]:
from langchain_community.document_loaders import PyPDFLoader

C:\Users\yashm\AppData\Local\Temp\ipykernel_3444\4175148793.py:1: DeprecationWarning: `langchain-community` is being sunset and is no longer actively maintained. See https://github.com/langchain-ai/langchain-community/issues/674 for details and migration guidance toward standalone integration packages.
  from langchain_community.document_loaders import PyPDFLoader


In [8]:
loader = PyPDFLoader("data/Diseases_in_RAG.pdf")
documents = loader.load()

In [9]:
%pip install pypdf

Defaulting to user installation because normal site-packages is not writeable
Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 25.3 -> 26.2.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [10]:
print("Pages : ", len(documents))

Pages :  6


In [11]:
for doc in documents[:5]:
    print(doc.page_content[:500])

Diseases in RAG & Their Solutions 
1. Hallucination 
• Problem 
o LLM generates incorrect or fabricated information. 
o Retrieved documents do not support the answers. 
• Solutions 
o Improve retrieval quality. 
o Use grounded prompting. 
o Generate citation-based answers. 
o Add confidence scoring. 
 
2. Poor Retrieval 
• Problem 
o Relevant documents are not retrieved. 
o Irrelevant documents are returned. 
• Solutions 
o Use better embedding models. 
o Apply Hybrid Search (Vector + Keyword). 
4. Lost in the Middle 
• Problem 
o Important information in the middle of the context is ignored. 
• Solutions 
o Place important chunks at the beginning. 
o Compress the context. 
o Reorder retrieved documents. 
o Use long-context LLMs. 
 
5. Chunking Problems 
• Problem 
o Important information gets split across chunks. 
o Chunks are too large or too small. 
• Solutions 
o Use semantic chunking. 
o Add chunk overlap. 
o Optimize chunk size. 
o Preserve document structure. 
 
6. Embedding Mis

In [12]:
print(doc.metadata)

{'producer': 'Microsoft® Word 2021', 'creator': 'Microsoft® Word 2021', 'creationdate': '2026-10-10T13:49:06+05:30', 'author': 'Yash Mishra', 'moddate': '2026-10-10T13:49:06+05:30', 'source': 'data/Diseases_in_RAG.pdf', 'total_pages': 6, 'page': 4, 'page_label': '5'}


## PHASE 2 : Chunking

In [13]:
from langchain_text_splitters import RecursiveCharacterTextSplitter

In [14]:
splitter = RecursiveCharacterTextSplitter(chunk_size = 300, chunk_overlap = 60)

chunks = splitter.split_documents(documents)

print("Original Pages : ", len(documents))
print("Chunks ", len(chunks))

Original Pages :  6
Chunks  20


## PHASE 3 : Embeddings

In [17]:
from langchain_huggingface import HuggingFaceEmbeddings

In [16]:
%pip install langchain_huggingface

Defaulting to user installation because normal site-packages is not writeable
Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 25.3 -> 26.2.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [18]:
from langchain_huggingface import HuggingFaceEmbeddings

In [20]:
embeddings = HuggingFaceEmbeddings(
    model_name = "sentence-transformers/all-MiniLM-L6-v2"
)

C:\Users\yashm\AppData\Roaming\Python\Python314\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
C:\Users\yashm\AppData\Roaming\Python\Python314\site-packages\huggingface_hub\file_download.py:149: UserWarning: `huggingface_hub` cache-system uses symlinks by default to efficiently store duplicated files but your machine does not support them in C:\Users\yashm\.cache\huggingface\hub\models--sentence-transformers--all-MiniLM-L6-v2. Caching files will still work but in a degraded version that might require more space on your disk. This warning can be disabled by setting the `HF_HUB_DISABLE_SYMLINKS_WARNING` environment variable. For more details, see https://huggingface.co/docs/huggingface_hub/how-to-cache#limitations.
To support symlinks on Windows, you either need to activate Developer Mode or to run Python as an adminis

In [21]:
%pip install sentence-transformers

Defaulting to user installation because normal site-packages is not writeable
Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 25.3 -> 26.2.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [22]:
vector = embeddings.embed_query("what is the solution to solve this problem : Important information in the middle of the context is ignored ?")
print(vector, len(vector))

[0.06898272782564163, 0.18987137079238892, 0.014661192893981934, -0.059552691876888275, 0.09305641055107117, 0.007184947840869427, 0.11080971360206604, 0.010786164551973343, 0.015222729183733463, -0.02235439047217369, 0.038349252194166183, -0.01083093136548996, 0.028413409367203712, 0.00538848340511322, -0.008177066221833229, 0.058019548654556274, 0.03504980355501175, 0.02500731311738491, -0.14292609691619873, -0.04903148114681244, 0.0078719612210989, 0.005376010201871395, 0.026304397732019424, 0.02103278413414955, 0.009353013709187508, 0.0986589789390564, -0.0296152476221323, 0.018142379820346832, 0.04127347841858864, -0.020251844078302383, 0.054198384284973145, 0.12528391182422638, 0.08823993802070618, 0.02855958789587021, -0.04001133143901825, 0.06345248967409134, -0.09730982780456543, 0.03518925607204437, 0.06243492290377617, -0.032633375376462936, -0.012606234289705753, 0.06438124179840088, 0.023043379187583923, 0.04537632688879967, 0.008730784058570862, 0.015845434740185738, 0.02

## PHASE 4 : Chroma Vector DataBase

In [23]:
%pip install langchain_chroma

Defaulting to user installation because normal site-packages is not writeable
Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 25.3 -> 26.2.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [24]:
from langchain_chroma import Chroma

In [25]:
vectorStore = Chroma.from_documents(
    documents = chunks,
    embedding = embeddings,
    persist_directory = "./chroma_db"
)